In [1]:
from pathlib import Path
import subprocess
import sys

from instance_reader import read_instance
from initial_solution import greedy_initial_solution


from alns import (
    SAParams, PenaltyParams, ALNSParams, run_alns,
    RandomRemoval, WorstDensityRemoval, SkillScarcityRemoval,
    RelatedRemoval, WorstDetourRemoval, WorstDetourRemovalV2,
    ShawRelatedRemoval, RouteRemoval, TimeWindowSegmentRemoval,
    GreedyBestInsertionRepair, Regret2InsertionRepair,
    SequentialCheapestInsertionRepair, NoisyGreedyInsertionRepair,
    ScarceSkillFirstRepair,
)

In [2]:
instance_path = Path("dataset/skillvrp_n400_v10_s8_k2.0_10.txt")
checker_path = Path("checker.py")

solution_dir = Path("solutions")
solution_dir.mkdir(parents=True, exist_ok=True)

solution_path = solution_dir / "solution.txt"

In [3]:
inst = read_instance(str(instance_path))

In [4]:
start_solution = greedy_initial_solution(
    inst,
    strategy="multi_start",
    verbose=True,
)

print("Initial objective:", start_solution.objective)

Greedy strategy: hybrid
  Objective: 9750
  Served customers: 128/400
Greedy strategy: profit
  Objective: 10426
  Served customers: 122/400
Greedy strategy: density
  Objective: 10255
  Served customers: 130/400
Greedy strategy: fewest_vehicles
  Objective: 10035
  Served customers: 122/400

Selected greedy strategy: profit
Best objective: 10426
Served customers: 122/400
Initial objective: 10426


In [5]:
sa = SAParams(
    initial_temperature=100.0,
    min_temperature=0.001,
    cooling_rate=0.9995,
    reheat_factor=0.75,
)

penalties = PenaltyParams(
    time_window_penalty=20.0,
    shift_penalty=20.0,
    skill_penalty=10000.0,
)

params = ALNSParams(
    random_seed=1,
    segment_length=100,
    no_accept_limit=500,
    reaction_factor=0.20,
    min_operator_weight=0.05,
    score_global_best=25.0,
    score_better_current=10.0,
    score_accepted=3.0,
    score_rejected=0.0,
    time_cost_alpha=0.5,
    time_scale_seconds=0.01,
    verbose=True,
    polish_interval_seconds=0.0,   # >0 aktiviert den v2 Or-opt-Polish auf neuen Best-Lösungen
)


destroy_operators = [
    RandomRemoval(
        fraction=0.15,
        min_remove=1,
        max_remove=50,
        initial_weight=1.0,
    ),
    WorstDensityRemoval(
        fraction=0.15,
        min_remove=1,
        max_remove=50,
        noise=0.05,
        initial_weight=1.0,
    ),
    SkillScarcityRemoval(
        fraction=0.15,
        min_remove=1,
        max_remove=50,
        noise=0.05,
        initial_weight=1.0,
    ),
    # --- aus alns(1), vorher nicht in der Liste ---
    RelatedRemoval(
        fraction=0.20,
        min_remove=1,
        max_remove=50,
        bias=4.0,
        initial_weight=1.0,
    ),
    WorstDetourRemoval(
        fraction=0.15,
        min_remove=1,
        max_remove=50,
        bias=4.0,
        initial_weight=1.0,
    ),
    # --- neu gemergt aus alns.py ---
    WorstDetourRemovalV2(
        fraction=0.15,
        min_remove=1,
        max_remove=50,
        noise=0.05,
        selection_bias=3.0,
        initial_weight=1.0,
    ),
    ShawRelatedRemoval(
        fraction=0.20,
        min_remove=1,
        max_remove=50,
        p_determinism=4.0,
        w_distance=0.5,
        w_time=0.25,
        w_skill=0.25,
        initial_weight=1.0,
    ),
    RouteRemoval(
        max_routes=2,
        selection_bias=3.0,
        initial_weight=1.0,
    ),
    TimeWindowSegmentRemoval(
        window_fraction=0.15,
        max_remove=50,
        initial_weight=1.0,
    ),
]

repair_operators = [
    GreedyBestInsertionRepair(
        extra_unserved_limit=100,
        max_insertions=None,
        min_delta_score=0.0,
        initial_weight=1.0,
    ),
    Regret2InsertionRepair(
        extra_unserved_limit=100,
        max_insertions=None,
        min_delta_score=0.0,
        initial_weight=1.0,
    ),
    # --- aus alns(1), vorher nicht in der Liste ---
    SequentialCheapestInsertionRepair(
        extra_unserved_limit=100,
        order="profit",
        initial_weight=1.0,
        noise_amp=0.0,
    ),
    # --- neu gemergt aus alns.py ---
    NoisyGreedyInsertionRepair(
        extra_unserved_limit=100,
        max_insertions=None,
        min_delta_score=0.0,
        noise=0.15,
        initial_weight=1.0,
    ),
    ScarceSkillFirstRepair(
        extra_unserved_limit=100,
        min_delta_score=0.0,
        initial_weight=1.0,
    ),
]

In [6]:
result = run_alns(
    runtime=30.0,
    inst=inst,
    start_solution=start_solution,
    sa=sa,
    params=params,
    penalties=penalties,
    destroy_operators=destroy_operators,
    repair_operators=repair_operators,
)

print()
print("ALNS finished")
print("Best objective:", result.best_evaluation.profit)
print("Best penalty:", result.best_evaluation.penalty)
print("Best feasible:", result.best_evaluation.feasible)
print("Iterations:", result.iterations)
print("Restarts:", result.restarts)

iter=    100 | best=   12098 | current_score=  12023.00 | current_feasible=True | T= 73.2538
iter=    200 | best=   12256 | current_score=  12166.00 | current_feasible=True | T= 54.4728
iter=    300 | best=   12329 | current_score=  12289.00 | current_feasible=True | T= 42.1518
iter=    400 | best=   12480 | current_score=  12452.00 | current_feasible=True | T= 31.1818
iter=    500 | best=   12677 | current_score=  12622.00 | current_feasible=True | T= 23.2584
iter=    600 | best=   12677 | current_score=  12442.00 | current_feasible=True | T= 18.2569
iter=    700 | best=   12677 | current_score=  12500.00 | current_feasible=True | T= 14.0257
iter=    800 | best=   12677 | current_score=  12406.00 | current_feasible=True | T= 11.0204
iter=    900 | best=   12677 | current_score=  12509.00 | current_feasible=True | T=  8.2164
iter=   1000 | best=   12677 | current_score=  12557.00 | current_feasible=True | T=  6.4203
iter=   1100 | best=   12696 | current_score=  12696.00 | current_feas

In [7]:
result.best_solution.write_for_checker(str(solution_path))

In [8]:
checker_result = subprocess.run(
    [
        sys.executable,
        str(checker_path),
        str(instance_path),
        str(solution_path),
    ],
    capture_output=True,
    text=True,
)

print()
print("Checker stdout:")
print(checker_result.stdout)

if checker_result.stderr:
    print("Checker stderr:")
    print(checker_result.stderr)

print("Checker return code:", checker_result.returncode)


Checker stdout:
OK: feasible, profit = 12818, visited 164/400 customers

Checker return code: 0


## Benchmark über alle Instanzen

Die folgenden Zellen laufen `run_alns` über **alle** Instanzen im `dataset/`-Ordner und
sammeln die Ergebnisse. Jede Instanz bekommt über `make_operators(inst)` ein **frisches**
Operator-Set (Operatoren sind zustandsbehaftet — Gewichte/Zähler dürfen nicht zwischen
Instanzen überlaufen). Danach folgt eine Zelle zur Operator-Nutzung/-Gewichtung.

In [9]:
# --- Factory: frisches Operator-Set pro Lauf -----------------------------
# Operatoren sind zustandsbehaftet (weight, total_uses, segment_*). Fuer jeden
# run_alns-Aufruf brauchen wir ein unabhaengiges, gewichts-zurueckgesetztes Set,
# sonst bluten adaptierte Gewichte und Timing zwischen den Instanzen durch.
# max_remove skaliert mit der Instanzgroesse (Bruchteil + absolute Obergrenze).

def make_operators(inst):
    max_remove = max(5, inst.num_customers // 3)

    destroy_operators = [
        RandomRemoval(
            fraction=0.15, min_remove=1, max_remove=max_remove, initial_weight=1.0,
        ),
        WorstDensityRemoval(
            fraction=0.15, min_remove=1, max_remove=max_remove, noise=0.05,
            initial_weight=1.0,
        ),
        SkillScarcityRemoval(
            fraction=0.15, min_remove=1, max_remove=max_remove, noise=0.05,
            initial_weight=1.0,
        ),
        RelatedRemoval(
            fraction=0.20, min_remove=1, max_remove=max_remove, bias=4.0,
            initial_weight=1.0,
        ),
        WorstDetourRemoval(
            fraction=0.15, min_remove=1, max_remove=max_remove, bias=4.0,
            initial_weight=1.0,
        ),
        WorstDetourRemovalV2(
            fraction=0.15, min_remove=1, max_remove=max_remove, noise=0.05,
            selection_bias=3.0, initial_weight=1.0,
        ),
        ShawRelatedRemoval(
            fraction=0.20, min_remove=1, max_remove=max_remove, p_determinism=4.0,
            w_distance=0.5, w_time=0.25, w_skill=0.25, initial_weight=1.0,
        ),
        RouteRemoval(
            max_routes=2, selection_bias=3.0, initial_weight=1.0,
        ),
        TimeWindowSegmentRemoval(
            window_fraction=0.15, max_remove=max_remove, initial_weight=1.0,
        ),
    ]

    repair_operators = [
        GreedyBestInsertionRepair(
            extra_unserved_limit=100, max_insertions=None, min_delta_score=0.0,
            initial_weight=1.0,
        ),
        Regret2InsertionRepair(
            extra_unserved_limit=100, max_insertions=None, min_delta_score=0.0,
            initial_weight=1.0,
        ),
        SequentialCheapestInsertionRepair(
            extra_unserved_limit=100, order="profit", initial_weight=1.0,
            noise_amp=0.0,
        ),
        NoisyGreedyInsertionRepair(
            extra_unserved_limit=100, max_insertions=None, min_delta_score=0.0,
            noise=0.15, initial_weight=1.0,
        ),
        ScarceSkillFirstRepair(
            extra_unserved_limit=100, min_delta_score=0.0, initial_weight=1.0,
        ),
    ]

    return destroy_operators, repair_operators

In [10]:
# --- Benchmark: run_alns ueber alle Instanzen ----------------------------
import re
import time
import pandas as pd
from pathlib import Path

DATASET_DIR = Path("dataset")
RUNTIME_PER_INSTANCE = 15.0   # Sekunden pro Instanz -- an Zeitbudget anpassen


def _n_customers(path: Path) -> int:
    m = re.search(r"_n(\d+)_", path.name)
    return int(m.group(1)) if m else 0


# Nach Kundenzahl sortiert (klein -> gross), damit instances[-1] die groesste ist.
instances = sorted(DATASET_DIR.glob("skillvrp_*.txt"), key=_n_customers)

# Gemeinsame SA/Penalty/Param-Konfiguration fuer den Benchmark.
bench_sa = SAParams(
    initial_temperature=100.0, min_temperature=0.001,
    cooling_rate=0.9995, reheat_factor=0.75,
)
bench_penalties = PenaltyParams(
    time_window_penalty=20.0, shift_penalty=20.0, skill_penalty=10000.0,
)
bench_params = ALNSParams(
    random_seed=1, segment_length=100, no_accept_limit=500,
    reaction_factor=0.20, min_operator_weight=0.05,
    score_global_best=25.0, score_better_current=10.0,
    score_accepted=3.0, score_rejected=0.0,
    time_cost_alpha=0.5, time_scale_seconds=0.01,
    verbose=False, polish_interval_seconds=0.0,
)

rows = []
for path in instances:
    inst_b = read_instance(str(path))
    start_b = greedy_initial_solution(inst_b, strategy="multi_start", verbose=False)
    destroy_b, repair_b = make_operators(inst_b)   # frisches Set pro Instanz

    t0 = time.perf_counter()
    res = run_alns(
        runtime=RUNTIME_PER_INSTANCE,
        inst=inst_b,
        start_solution=start_b,
        sa=bench_sa,
        params=bench_params,
        penalties=bench_penalties,
        destroy_operators=destroy_b,
        repair_operators=repair_b,
    )
    elapsed = time.perf_counter() - t0

    greedy_profit = start_b.objective
    alns_profit = res.best_evaluation.profit
    improvement = (alns_profit - greedy_profit) / greedy_profit * 100 if greedy_profit else 0.0

    rows.append({
        "instance": path.name,
        "n": inst_b.num_customers,
        "greedy": greedy_profit,
        "alns": alns_profit,
        "improve_%": improvement,
        "feasible": res.best_evaluation.feasible,
        "iters": res.iterations,
        "restarts": res.restarts,
        "sec": elapsed,
    })

    print(f"{path.name:42s} greedy {greedy_profit:>7} -> alns {alns_profit:>7}  "
          f"(+{improvement:5.1f}%, {res.iterations} it, {res.restarts} rs)")

bench_df = pd.DataFrame(rows)
print(f"\nDurchschnittliche Verbesserung: {bench_df['improve_%'].mean():.1f}%  |  "
      f"alle feasible: {bench_df['feasible'].all()}")
bench_df.round(2)

skillvrp_n50_v2_s3_k0.0_1.txt              greedy    1501 -> alns    1802  (+ 20.1%, 40507 it, 0 rs)
skillvrp_n75_v2_s4_k0.5_2.txt              greedy    2058 -> alns    2776  (+ 34.9%, 19718 it, 0 rs)
skillvrp_n100_v2_s5_k1.0_3.txt             greedy    2292 -> alns    2975  (+ 29.8%, 14970 it, 0 rs)
skillvrp_n125_v3_s6_k1.5_4.txt             greedy    3697 -> alns    4267  (+ 15.4%, 9825 it, 0 rs)
skillvrp_n150_v3_s8_k2.0_5.txt             greedy    4065 -> alns    4857  (+ 19.5%, 6997 it, 0 rs)
skillvrp_n200_v5_s3_k0.0_6.txt             greedy    5504 -> alns    7222  (+ 31.2%, 3933 it, 0 rs)
skillvrp_n250_v6_s4_k0.5_7.txt             greedy    7845 -> alns    9637  (+ 22.8%, 3772 it, 0 rs)
skillvrp_n300_v7_s5_k1.0_8.txt             greedy    8331 -> alns   11372  (+ 36.5%, 2980 it, 0 rs)
skillvrp_n350_v8_s6_k1.5_9.txt             greedy    9708 -> alns   12613  (+ 29.9%, 2697 it, 0 rs)
skillvrp_n400_v10_s8_k2.0_10.txt           greedy   10426 -> alns   12795  (+ 22.7%, 2332 it, 0 r

,instance,n,greedy,alns,improve_%,feasible,iters,restarts,sec
0,skillvrp_n50_v2_s3_k0.0_1.txt,50,1501,1802,20.05,True,40507,0,15.0
1,skillvrp_n75_v2_s4_k0.5_2.txt,75,2058,2776,34.89,True,19718,0,15.0
2,skillvrp_n100_v2_s5_k1.0_3.txt,100,2292,2975,29.80,True,14970,0,15.0
3,skillvrp_n125_v3_s6_k1.5_4.txt,125,3697,4267,15.42,True,9825,0,15.0
4,skillvrp_n150_v3_s8_k2.0_5.txt,150,4065,4857,19.48,True,6997,0,15.0
5,skillvrp_n200_v5_s3_k0.0_6.txt,200,5504,7222,31.21,True,3933,0,15.0
6,skillvrp_n250_v6_s4_k0.5_7.txt,250,7845,9637,22.84,True,3772,0,15.0
7,skillvrp_n300_v7_s5_k1.0_8.txt,300,8331,11372,36.50,True,2980,0,15.0
8,skillvrp_n350_v8_s6_k1.5_9.txt,350,9708,12613,29.92,True,2697,0,15.0
9,skillvrp_n400_v10_s8_k2.0_10.txt,400,10426,12795,22.72,True,2332,0,15.0


In [11]:
# --- Operator-Nutzung und -Gewichte --------------------------------------
# Ein einzelner, laengerer Lauf auf einer Instanz; danach zeigen wir, welche
# Operatoren die adaptiven Gewichte auf sich gezogen haben (= was tatsaechlich
# beitraegt). Eigene Variablennamen (op_*), damit die Zellen oben nicht
# ueberschrieben werden.
import pandas as pd

OP_STATS_INSTANCE = instances[-1]    # groesste Instanz; nach Belieben aendern
OP_STATS_RUNTIME = 15.0

op_inst = read_instance(str(OP_STATS_INSTANCE))
op_start = greedy_initial_solution(op_inst, verbose=False)
op_destroy, op_repair = make_operators(op_inst)

served0 = op_inst.num_customers - len(op_start.unserved)
avg_profit = op_start.objective / served0 if served0 else 1.0

op_result = run_alns(
    runtime=OP_STATS_RUNTIME,
    inst=op_inst,
    start_solution=op_start,
    sa=SAParams(initial_temperature=max(1.0, 0.8 * avg_profit),
                min_temperature=0.01, cooling_rate=0.9997, reheat_factor=0.5),
    params=ALNSParams(random_seed=42, segment_length=100, no_accept_limit=3000,
                      reaction_factor=0.2, min_operator_weight=0.1,
                      score_global_best=10.0, score_better_current=5.0,
                      score_accepted=2.0, score_rejected=0.0,
                      time_cost_alpha=1.0, time_scale_seconds=0.01,
                      verbose=False),
    penalties=PenaltyParams(10.0, 10.0, 10_000.0),
    destroy_operators=op_destroy,
    repair_operators=op_repair,
)

print(f"{OP_STATS_INSTANCE.name}: Greedy {op_start.objective} -> "
      f"ALNS {op_result.best_evaluation.profit}  "
      f"({op_result.iterations} Iterationen, {op_result.restarts} Restarts)\n")

destroy_df = pd.DataFrame(op_result.destroy_summary).sort_values("weight", ascending=False)
repair_df = pd.DataFrame(op_result.repair_summary).sort_values("weight", ascending=False)

print("Destroy-Operatoren:")
display(destroy_df.round(4))
print("Repair-Operatoren:")
display(repair_df.round(4))

skillvrp_n1000_v25_s8_k2.0_20.txt: Greedy 30193 -> ALNS 35569  (415 Iterationen, 0 Restarts)

Destroy-Operatoren:


,name,weight,uses,avg_time,total_time
5,worst_detour_removal_v2,1.1198,55,0.0006,0.0311
4,worst_detour_removal,0.9075,66,0.0005,0.0329
2,skill_scarcity_removal,0.6938,51,0.0008,0.0386
0,random_removal,0.6696,42,0.0003,0.0130
1,worst_density_removal,0.6525,46,0.0004,0.0176
7,route_removal,0.6448,46,0.0001,0.0058
8,time_segment_removal,0.5118,43,0.0004,0.0160
3,related_removal,0.4598,39,0.0004,0.0137
6,shaw_related_removal,0.3277,27,0.0060,0.1614


Repair-Operatoren:


,name,weight,uses,avg_time,total_time
2,sequential_insertion_profit,0.4911,93,0.0131,1.2189
0,greedy_best_insertion,0.4830,90,0.0455,4.0974
4,scarce_skill_first_insertion,0.4273,91,0.0344,3.1316
3,noisy_greedy_insertion,0.3579,77,0.0421,3.2451
1,regret2_insertion,0.3277,64,0.0464,2.9675
